In [1]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"
train_dataset = load_dataset(dataset_name, split="train")
test_dataset = load_dataset(dataset_name, split="test")
sample = train_dataset[0]

### Process

In [4]:
import json
from typing import Any
import torch

In [3]:
sample['ground_truth']

'{"bill_to": {"name": "PT Maju Teknologi", "address": "Gg. Tebet Barat Dalam No. 40", "city": "Bontang, Indonesia", "phone": "+62 823-9935-2424", "email": "finance@majuteknologi.com"}, "invoice_date": "1 February 2025", "invoice_no": "INV-2025-30496", "from_to": {"name": "PT Solusi Nusantara", "address": "Gang Antapani Lama No. 34", "city": "Meulaboh, Indonesia", "phone": "+62 885-7912-1520", "email": "billing@solusinusantara.com"}, "items": [{"description": "DevOps Services", "qty": 1, "price": 3800000, "total": 3800000}, {"description": "UI/UX Design", "qty": 9, "price": 2900000, "total": 26100000}, {"description": "UI/UX Design", "qty": 8, "price": 4000000, "total": 32000000}, {"description": "Cloud Infrastructure", "qty": 1, "price": 1250000, "total": 1250000}, {"description": "AI Development", "qty": 7, "price": 2400000, "total": 16800000}, {"description": "Cloud Infrastructure", "qty": 3, "price": 1600000, "total": 4800000}], "subtotal": 84750000, "note": "Thank you for your busi

In [5]:
ground_truth = json.loads(sample['ground_truth'])

In [6]:
ground_truth

{'bill_to': {'name': 'PT Maju Teknologi',
  'address': 'Gg. Tebet Barat Dalam No. 40',
  'city': 'Bontang, Indonesia',
  'phone': '+62 823-9935-2424',
  'email': 'finance@majuteknologi.com'},
 'invoice_date': '1 February 2025',
 'invoice_no': 'INV-2025-30496',
 'from_to': {'name': 'PT Solusi Nusantara',
  'address': 'Gang Antapani Lama No. 34',
  'city': 'Meulaboh, Indonesia',
  'phone': '+62 885-7912-1520',
  'email': 'billing@solusinusantara.com'},
 'items': [{'description': 'DevOps Services',
   'qty': 1,
   'price': 3800000,
   'total': 3800000},
  {'description': 'UI/UX Design',
   'qty': 9,
   'price': 2900000,
   'total': 26100000},
  {'description': 'UI/UX Design',
   'qty': 8,
   'price': 4000000,
   'total': 32000000},
  {'description': 'Cloud Infrastructure',
   'qty': 1,
   'price': 1250000,
   'total': 1250000},
  {'description': 'AI Development',
   'qty': 7,
   'price': 2400000,
   'total': 16800000},
  {'description': 'Cloud Infrastructure',
   'qty': 3,
   'price': 160

In [7]:
def json2token(
    obj: Any,
    sort_json_key: bool = False,
    all_special_tokens: list[str] | None = None,
):
    tokens_to_add = set()

    if all_special_tokens is None:
        all_special_tokens = []

    if type(obj) == dict:
        if len(obj) == 1 and "text_sequence" in obj:
            return obj["text_sequence"], tokens_to_add

        output = ""

        if sort_json_key:
            keys = sorted(obj.keys(), reverse=True)
        else:
            keys = obj.keys()

        for k in keys:
            tokens_to_add.update([
                f"<s_{k}>",
                f"</s_{k}>",
            ])

            value, child_tokens = json2token(
                obj[k],
                sort_json_key,
                all_special_tokens,
            )

            tokens_to_add.update(child_tokens)

            output += (
                f"<s_{k}>"
                + value
                + f"</s_{k}>"
            )

        return output, tokens_to_add

    elif type(obj) == list:
        outputs = []

        for item in obj:
            value, child_tokens = json2token(
                item,
                sort_json_key,
                all_special_tokens,
            )

            outputs.append(value)
            tokens_to_add.update(child_tokens)

        return r"<sep/>".join(outputs), tokens_to_add

    else:
        obj = str(obj)

        if f"<{obj}/>" in all_special_tokens:
            obj = f"<{obj}/>"

        return obj, tokens_to_add

In [8]:
output, list_of_tokens = json2token(
    ground_truth,
    sort_json_key=False,
    all_special_tokens=[]
)

In [9]:
output

'<s_bill_to><s_name>PT Maju Teknologi</s_name><s_address>Gg. Tebet Barat Dalam No. 40</s_address><s_city>Bontang, Indonesia</s_city><s_phone>+62 823-9935-2424</s_phone><s_email>finance@majuteknologi.com</s_email></s_bill_to><s_invoice_date>1 February 2025</s_invoice_date><s_invoice_no>INV-2025-30496</s_invoice_no><s_from_to><s_name>PT Solusi Nusantara</s_name><s_address>Gang Antapani Lama No. 34</s_address><s_city>Meulaboh, Indonesia</s_city><s_phone>+62 885-7912-1520</s_phone><s_email>billing@solusinusantara.com</s_email></s_from_to><s_items><s_description>DevOps Services</s_description><s_qty>1</s_qty><s_price>3800000</s_price><s_total>3800000</s_total><sep/><s_description>UI/UX Design</s_description><s_qty>9</s_qty><s_price>2900000</s_price><s_total>26100000</s_total><sep/><s_description>UI/UX Design</s_description><s_qty>8</s_qty><s_price>4000000</s_price><s_total>32000000</s_total><sep/><s_description>Cloud Infrastructure</s_description><s_qty>1</s_qty><s_price>1250000</s_price><s

In [10]:
list_of_tokens

{'</s_account_name>',
 '</s_account_number>',
 '</s_address>',
 '</s_bank>',
 '</s_bill_to>',
 '</s_city>',
 '</s_description>',
 '</s_email>',
 '</s_from_to>',
 '</s_invoice_date>',
 '</s_invoice_no>',
 '</s_items>',
 '</s_name>',
 '</s_note>',
 '</s_payment_information>',
 '</s_phone>',
 '</s_price>',
 '</s_qty>',
 '</s_subtotal>',
 '</s_total>',
 '<s_account_name>',
 '<s_account_number>',
 '<s_address>',
 '<s_bank>',
 '<s_bill_to>',
 '<s_city>',
 '<s_description>',
 '<s_email>',
 '<s_from_to>',
 '<s_invoice_date>',
 '<s_invoice_no>',
 '<s_items>',
 '<s_name>',
 '<s_note>',
 '<s_payment_information>',
 '<s_phone>',
 '<s_price>',
 '<s_qty>',
 '<s_subtotal>',
 '<s_total>'}

In [11]:
from scripts.tokenizer import get_default_tokenizer

In [12]:
tokenizer = get_default_tokenizer()

In [13]:
len(tokenizer)

57527

In [20]:
def process_ground_truth(tokenizer, ground_truth, sort_json_key = False, task_start_token="<s>"):
    ground_truth = json.loads(ground_truth)
    output, list_of_tokens = json2token(
        ground_truth,
        sort_json_key=sort_json_key,
        all_special_tokens=tokenizer.all_special_tokens
    )

    gt_token_sequence = task_start_token + output + tokenizer.eos_token
    
    added_tokens = set()
    added_tokens.update(list_of_tokens)
    return gt_token_sequence, sorted(added_tokens)

In [29]:
task_start_token = '<s_cord-v2>'
prompt_end_token= '<s_cord-v2>'

In [30]:
gt_token_sequence, added_tokens = process_ground_truth(tokenizer, sample["ground_truth"], sort_json_key = False, task_start_token=task_start_token)

In [31]:
gt_token_sequence

'<s_cord-v2><s_bill_to><s_name>PT Maju Teknologi</s_name><s_address>Gg. Tebet Barat Dalam No. 40</s_address><s_city>Bontang, Indonesia</s_city><s_phone>+62 823-9935-2424</s_phone><s_email>finance@majuteknologi.com</s_email></s_bill_to><s_invoice_date>1 February 2025</s_invoice_date><s_invoice_no>INV-2025-30496</s_invoice_no><s_from_to><s_name>PT Solusi Nusantara</s_name><s_address>Gang Antapani Lama No. 34</s_address><s_city>Meulaboh, Indonesia</s_city><s_phone>+62 885-7912-1520</s_phone><s_email>billing@solusinusantara.com</s_email></s_from_to><s_items><s_description>DevOps Services</s_description><s_qty>1</s_qty><s_price>3800000</s_price><s_total>3800000</s_total><sep/><s_description>UI/UX Design</s_description><s_qty>9</s_qty><s_price>2900000</s_price><s_total>26100000</s_total><sep/><s_description>UI/UX Design</s_description><s_qty>8</s_qty><s_price>4000000</s_price><s_total>32000000</s_total><sep/><s_description>Cloud Infrastructure</s_description><s_qty>1</s_qty><s_price>1250000<

In [24]:
# added_tokens

In [25]:
def get_train_inputs(tokenizer, processed_parse, prompt_end_token_id, max_length = 768, ignore_id = -100):
    input_ids = tokenizer(
        processed_parse,
        add_special_tokens=False,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )["input_ids"].squeeze(0)
    
    labels = input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = ignore_id 
    labels[: torch.nonzero(labels == prompt_end_token_id).sum() + 1] = ignore_id    
    return input_ids, labels

In [32]:
task_start_token = '<s_cord-v2>'
prompt_end_token= '<s_cord-v2>'

In [33]:
tokenizer.add_special_tokens({
    "additional_special_tokens": sorted(set([task_start_token, prompt_end_token]))
}) 

0

In [34]:
prompt_end_token_id = tokenizer.convert_tokens_to_ids(prompt_end_token)

In [35]:
input_ids, labels = get_train_inputs(tokenizer, gt_token_sequence, prompt_end_token_id)

In [36]:
input_ids

tensor([57527, 41040, 46192, 41403, 34796, 45246, 41403, 19616, 50454, 46192,
        41403,  4131, 34791, 26588, 34837, 36217, 10015, 43443,  6333, 40598,
        46192, 41403,  4131, 50454, 46192, 41403, 49557,  1933, 34791, 49933,
        17367, 39539, 10015, 41720, 37850, 39213,  9805, 40837, 39539,  2855,
        40598, 46192, 41403, 49557,  1933, 50454, 46192, 41403, 46222, 34791,
         6642, 40223, 35815, 38834, 40598, 46192, 41403, 46222, 50454, 46192,
        41403, 34436, 34791,  8053,  2166, 13563, 34835, 43137, 45675, 37168,
        27842, 40598, 46192, 41403, 34436, 50454, 46192, 41403,  5051, 23220,
        34791, 46140, 49682, 56981, 34715, 48463, 43443,  6333, 39539, 34874,
        40598, 46192, 41403,  5051, 23220, 16630, 46192, 41403, 34796, 45246,
        41403, 19616, 50454, 46192, 41403, 49827, 56599, 35732, 41403, 17164,
        34791, 56894, 26252, 51294, 40598, 46192, 41403, 49827, 56599, 35732,
        41403, 17164, 50454, 46192, 41403, 49827, 56599, 35732, 